# Sentiment Analysis

This notebook processes the language-classified sentences to extract sentiment for those in supported languages.

In [1]:
import pandas as pd
import os

# Google Colab Drive Mount & Checkpoint Configuration
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE_DIR = '/content/drive/MyDrive/Projects/letterboxd'
    print("Running in Google Colab. Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Using local directory for checkpoints.")
    DRIVE_BASE_DIR = './absa_project'

# Configure paths
file_path = os.path.join(DRIVE_BASE_DIR, 'letterboxd_sentences_with_language.parquet')

try:
    df = pd.read_parquet(file_path)
    print(f"Loaded {len(df)} sentences.")
except Exception as e:
    print(e)

Mounted at /content/drive
Running in Google Colab. Drive mounted successfully.
Loaded 425083 sentences.


In [2]:
# Filter for supported languages
supported_languages = ['en-US', 'nl-NL', 'de-DE', 'fr-FR', 'es-ES', 'it-IT']
df_supported = df[df['language'].isin(supported_languages)].copy()
print(f"{len(df_supported)} sentences in supported languages.")

399909 sentences in supported languages.


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Load sentiment model
model_name = 'nlptown/bert-base-multilingual-uncased-sentiment'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
sentiment_pipeline = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer, truncation=True, max_length=512)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:

import torch
from datasets import Dataset
from tqdm.auto import tqdm

print("Running sentiment analysis with HF Dataset and batching...")

# Convert pandas dataframe to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df_supported[['sentence_text']])

BATCH_SIZE = 128 # Adjust based on GPU

results_label = []
results_score = []

# Using pipeline natively with dataset and batch_size
from transformers.pipelines.pt_utils import KeyDataset

# Using pipeline natively with KeyDataset and batch_size
for out in tqdm(sentiment_pipeline(KeyDataset(hf_dataset, 'sentence_text'), batch_size=BATCH_SIZE, truncation=True, max_length=512), total=len(hf_dataset)):
    results_label.append(out['label'])
    results_score.append(out['score'])

df_supported['sentiment_label'] = results_label
df_supported['sentiment_score'] = results_score
df_supported.head()


Running sentiment analysis with HF Dataset and batching...


  0%|          | 0/399909 [00:00<?, ?it/s]

,review_id,film_id,film_name,genres,star_rating_num,sentence_id,sentence_text,is_truncated,language,sentiment_label,sentiment_score
0,1,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,1_0,my favourite part is how he doesn't give an ab...,False,en-US,5 stars,0.616550
1,2,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,2_0,tell me you wouldn't cry too if your son grows...,False,en-US,5 stars,0.254817
2,3,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,3_0,"I have a headache, but it's the best headache ...",False,en-US,5 stars,0.288259
3,4,1,Interstellar,"Adventure,Drama,Science Fiction",4.0,4_0,watched on my 13 inch macbook air just as chri...,False,en-US,5 stars,0.359607
4,5,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,5_0,""" It was you.",False,en-US,5 stars,0.301640


In [6]:
# Convert label (e.g., '5 stars') to numeric
df_supported['sentiment_numeric'] = df_supported['sentiment_label'].str.extract(r'(\d)').astype(int)
df_supported.head()

,review_id,film_id,film_name,genres,star_rating_num,sentence_id,sentence_text,is_truncated,language,sentiment_label,sentiment_score,sentiment_numeric
0,1,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,1_0,my favourite part is how he doesn't give an ab...,False,en-US,5 stars,0.616550,5
1,2,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,2_0,tell me you wouldn't cry too if your son grows...,False,en-US,5 stars,0.254817,5
2,3,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,3_0,"I have a headache, but it's the best headache ...",False,en-US,5 stars,0.288259,5
3,4,1,Interstellar,"Adventure,Drama,Science Fiction",4.0,4_0,watched on my 13 inch macbook air just as chri...,False,en-US,5 stars,0.359607,5
4,5,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,5_0,""" It was you.",False,en-US,5 stars,0.301640,5


In [8]:
# Save the results
out_path = os.path.join(DRIVE_BASE_DIR, 'letterboxd_sentences_with_sentiment.parquet')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_supported.to_parquet(out_path, index=False)
print("Saved sentiment analysis results.")

Saved sentiment analysis results.
